# 01. Ingesta, Auditoría de Calidad e Integración de Datos (HYPERAKTIV)

**Proyecto Final:** Análisis de Regresión y Correlación (ARC)  
**Autores:** Richie y Mariana  
**Fecha:** 16 de noviembre de 2026  
**Dataset:** HYPERAKTIV (Hicks et al., 2021) — `patient_info.csv`, `features.csv`, `CPT_II_ConnersContinuousPerformanceTest.csv`

**Objetivo:** Cargar las tres fuentes tabulares crudas, auditar su calidad e integridad por `ID` (duplicados, valores faltantes en `ASRS` y sesiones inválidas en `CPT-II`), integrarlas en una tabla analítica limpia y verificar que el tamaño muestral cumpla $n \ge 30$ para el modelado por Regresión Lineal Múltiple (MCO).

## 1. Configuración del Entorno y Rutas

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import os
from IPython.core.display_functions import display


In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

patient_info_path = RAW_DIR / "patient_info.csv"
features_path = RAW_DIR / "features.csv"
cpt_path = RAW_DIR / "CPT_II_ConnersContinuousPerformanceTest.csv"


patient_info_df = pd.read_csv(patient_info_path, sep=";", encoding="utf-8-sig")
features_df = pd.read_csv(features_path, sep=";", encoding="utf-8-sig")
cpt_df = pd.read_csv(cpt_path, sep=";", encoding="utf-8-sig")


## 2. Ingesta de Datos Crudos (Data Ingestion)

In [3]:

print(f"El Dataframe patient_info contiente {patient_info_df.shape[0]} filas y {patient_info_df.shape[1]} columnas.")
print(f"El Dataframe features contiente {features_df.shape[0]} filas y {features_df.shape[1]} columnas.")
print(f"El Dataframe cpt contiente {cpt_df.shape[0]} filas y {cpt_df.shape[1]} columnas.")
print("Primeras 5 observaciones de los Dataframes, en orden respectivo.")

print()
print("patient_info_df:")
display(patient_info_df.head())
print()
print("features_df:")
display(features_df.head())
print()
print("cpt_df:")
display(cpt_df.head())

El Dataframe patient_info contiente 103 filas y 33 columnas.
El Dataframe features contiente 116 filas y 788 columnas.
El Dataframe cpt contiente 99 filas y 786 columnas.
Primeras 5 observaciones de los Dataframes, en orden respectivo.

patient_info_df:


,ID,SEX,AGE,ACC,ACC_TIME,ACC_DAYS,HRV,HRV_TIME,HRV_HOURS,CPT_II,...,HADS_D,MED,MED_Antidepr,MED_Moodstab,MED_Antipsych,MED_Anxiety_Benzo,MED_Sleep,MED_Analgesics_Opioids,MED_Stimulants,filter_$
0,1,0,3,1,16:00:00,6.0,1,11:00:00,21.0,0,...,2.0,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2,0,4,1,10:54:00,6.8,0,NaN,NaN,1,...,7.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,3,1,2,1,15:28:00,7.2,1,15:25:00,21.0,1,...,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,4,1,3,0,NaN,NaN,1,16:55:00,22.0,1,...,6.0,1,1.0,NaN,NaN,1.0,NaN,1.0,NaN,0
4,5,1,1,1,14:24:00,5.9,1,16:00:00,12.0,1,...,5.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1



features_df:


,ID,ACC__variance_larger_than_standard_deviation,ACC__has_duplicate_max,ACC__has_duplicate_min,ACC__has_duplicate,ACC__sum_values,ACC__abs_energy,ACC__mean_abs_change,ACC__mean_change,ACC__mean_second_derivative_central,...,ACC__permutation_entropy__dimension_5__tau_1,ACC__permutation_entropy__dimension_6__tau_1,ACC__permutation_entropy__dimension_7__tau_1,ACC__query_similarity_count__query_None__threshold_0.0,"ACC__matrix_profile__feature_""min""__threshold_0.98","ACC__matrix_profile__feature_""max""__threshold_0.98","ACC__matrix_profile__feature_""mean""__threshold_0.98","ACC__matrix_profile__feature_""median""__threshold_0.98","ACC__matrix_profile__feature_""25""__threshold_0.98","ACC__matrix_profile__feature_""75""__threshold_0.98"
0,1,1.0,0.0,1.0,1.0,433530.0,1.228179e+08,34.574870,0.006254,-0.015752,...,3.120897,4.077141,4.880321,NaN,3.495156,15.586971,11.632054,12.013496,10.744060,13.176179
1,2,1.0,1.0,1.0,1.0,1572295.0,1.010897e+09,103.330694,0.000000,-0.002792,...,3.883058,5.149344,6.278021,NaN,3.054584,15.231546,9.369168,9.348224,8.285911,10.382951
2,3,1.0,0.0,1.0,1.0,5232087.0,6.869091e+09,251.845947,0.000000,-0.049439,...,4.364522,5.866384,7.260060,NaN,3.947638,19.595918,13.502457,13.651750,12.254941,14.778953
3,5,1.0,0.0,1.0,1.0,508325.0,1.018261e+08,46.477767,0.000000,-0.004494,...,4.096368,5.464033,6.680095,NaN,3.695875,14.341713,11.853373,12.124791,11.190192,12.813456
4,7,1.0,0.0,1.0,1.0,641357.0,1.545783e+08,34.675614,0.000000,-0.015340,...,3.415288,4.490762,5.442471,NaN,3.500962,22.090722,14.492647,14.989601,12.924897,16.403917



cpt_df:


,ID,Assessment Status,Assessment Duration,Type,LastTrial,Trial1,Response1,Trial2,Response2,Trial3,...,Raw Score HitRTIsi,Percent HitRTIsi,General TScore HitSEIsi,Adhd TScore HitSEIsi,Neuro TScore HitSEIsi,Raw Score HitSEIsi,Percent HitSEIsi,Adhd Confidence Index,Neuro Confidence Index,Old Overall Index
0,2,3,839999,6,360,11,-1,13,469,7,...,0.049592,0,48.021103,47.772537,43.224297,-0.021096,0,61.145496,58.749630,10.070000
1,3,3,839999,6,360,12,-1,0,371,2,...,0.064090,0,42.234097,38.263283,41.574574,-0.077351,0,86.871597,61.088688,4.440000
2,4,3,839999,6,360,14,-1,10,-1,3,...,0.077233,0,45.680782,42.289310,43.868061,-0.046142,0,99.900002,86.131813,18.340000
3,5,3,839999,6,360,12,459,7,289,12,...,0.085540,0,85.698700,70.469986,80.228539,0.410634,0,99.900002,99.900002,8.950001
4,7,3,839998,6,360,11,841,7,-1,5,...,0.024632,0,42.058998,43.349667,39.992119,-0.071200,0,32.291084,24.225414,0.000000


## 3. Auditoría de Calidad de Datos (Data Understanding & Quality Audit)

### 3.1 Unicidad de IDs y duplicados
- [ ] Verificar si hay `ID` nulos o duplicados en las 3 tablas.

In [4]:
patient_dups = patient_info_df.duplicated(subset="ID").value_counts()
features_dups = features_df.duplicated(subset="ID").value_counts()
cpt_dups = cpt_df.duplicated(subset="ID").value_counts()

print("IDs duplicados en patient_info_df: ")
print(patient_dups)

print()
print(f"IDs duplicados en features_df: \n{features_dups}")
print()

print()
print(f"IDs duplicados en cpt_df: \n{cpt_dups}")
print()

IDs duplicados en patient_info_df: 
False    103
Name: count, dtype: int64

IDs duplicados en features_df: 
False    116
Name: count, dtype: int64


IDs duplicados en cpt_df: 
False    99
Name: count, dtype: int64



> No hay valores duplicados en las tablas.

### 3.2 Valores Faltantes en la Variable Objetivo (ASRS)

- [ ] Contar cuántos pacientes tienen ASRS nulo en `patient_info`.

In [5]:
asrs_null = patient_info_df["ASRS"].isna().sum()
asrs_null

np.int64(7)

> 7 pacientes tienen `ASRS` nulo.

### 3.3 Regla de Calidad Clínica en CPT-II (Raw Score Ommisions)

- [ ] Revisar cuántos registros de CPT-II tienen `Raw Score Omissions >= 300` (según el paper de HYPERAKTIV son sesiones inválidas por desconexión de harware o abandono de la prueba).

In [6]:
cpt_df[cpt_df["Raw Score Omissions"] >= 300]

,ID,Assessment Status,Assessment Duration,Type,LastTrial,Trial1,Response1,Trial2,Response2,Trial3,...,Raw Score HitRTIsi,Percent HitRTIsi,General TScore HitSEIsi,Adhd TScore HitSEIsi,Neuro TScore HitSEIsi,Raw Score HitSEIsi,Percent HitSEIsi,Adhd Confidence Index,Neuro Confidence Index,Old Overall Index
11,15,3,839998,6,360,3,-1,9,-1,11,...,0.0,-1,0.0,0.0,0.0,0.0,-1,0.0,0.0,4.44
93,101,3,840000,6,360,0,-1,7,-1,8,...,0.0,-1,0.0,0.0,0.0,0.0,-1,0.0,0.0,4.44


> Existen 2 observaciones con sesiones invalidas en cpt

## 4. Limpieza e Integración de Tablas (Data Preparation & Integration)

### 4.1 Filtrado de Sesiones Inválidas y Selección de Columnas Resumen

- [ ] Filtrar `Raw Score Omissions < 300` en CPT-II


In [7]:
cpt_df = cpt_df[cpt_df["Raw Score Omissions"] < 300]

print("Se han eliminado las sesiones inválidas correctamente...")

Se han eliminado las sesiones inválidas correctamente...


- [ ] Opcional: excluir columnas de detalle por ensayo (`Trial..., Response...`) para quedarnos únicamente con las métricas clínicas agregadas.


In [8]:
summary_cols = [
    col for col in cpt_df.columns
    if not col.startswith(("Trial", "Response"))
]
cpt_summary_df = cpt_df[summary_cols].copy()

print(f"Columnas originales de cpt: {cpt_df.shape[1]} -> Columnas resumen: {cpt_summary_df.shape[1]}")


Columnas originales de cpt: 786 -> Columnas resumen: 66



### 4.2 Integración por ID (Inner Join)

- [ ] Unir las 3 tablas por ID (`how="inner`) y descartar filas con `ASRS` nulo.




- [ ] Verificar el tamaño muestral final $(n,p)$ y confirmar que $n \leq 30$ (por requisito de la rúbrica).

In [9]:
merged_df = (
    patient_info_df
    .merge(cpt_summary_df, on="ID", how="inner")
    .merge(features_df, on="ID", how="inner")
    .dropna(subset=["ASRS"])
    .reset_index(drop=True)
)

print(f"Dataset integrado: {merged_df.shape[0]} filas y {merged_df.shape[1]} columnas.")

Dataset integrado: 75 filas y 885 columnas.


> Cumple con $n\geq30$

## 5. Validación Post-Integración y Exportación

- [ ] Revisar `.dtypes`, nulos residuales y estadísticas descriptivas de `ASRS` (`describe()`).


In [10]:
print("Tipos de datos en merged_df:")
display(merged_df.dtypes.value_counts())

print("\nEstadísticas descriptivas de ASRS:")
display(merged_df['ASRS'].describe())

Tipos de datos en merged_df:


float64    852
int64       31
str          2
Name: count, dtype: int64


Estadísticas descriptivas de ASRS:


count    75.000000
mean     41.386667
std      13.904008
min       6.000000
25%      33.000000
50%      42.000000
75%      52.500000
max      68.000000
Name: ASRS, dtype: float64

- [ ] Guardar el resultado en `../data/processed/hyperaktiv_arc_clean.csv` (`index=False`).

In [11]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DIR / "hyperaktiv_arc_clean.csv"
merged_df.to_csv(output_path, index=False)

print(f"\nArchivo guardado en {output_path}")
output_path.exists()


Archivo guardado en ../data/processed/hyperaktiv_arc_clean.csv


True